In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
%sql
use catalog dev_catalog

## function to take delta tables from bronze schema

In [0]:
def bronze_to_silver(path,label:str):
        df=spark.table(path)
        print(f'{label} loaded successfully..')
        return df
silver_tb_1=bronze_to_silver('dev_catalog.bronze_schema.bank','file1')
silver_tb_2=bronze_to_silver('dev_catalog.bronze_schema.bigmartsales','file2')

file1 loaded successfully..
file2 loaded successfully..


Parameters for this project
- Row count
- Column Difference
- Null Values
- Statistical Difference
- Output report table

In [0]:
def shape(tb,table_name:str):
    """
    This function will tell you how many rows and column are there in the table
    """
    print(f'The table {table_name} looks like this...')
    print(f'No of rows: {tb.count()}')
    print(f'No of columns: {len(tb.columns)}')
shape(silver_tb_1,'bank')
shape(silver_tb_2,'bigmartsales')

The table bank looks like this...
No of rows: 4521
No of columns: 17
The table bigmartsales looks like this...
No of rows: 8523
No of columns: 12


In [0]:
def column_diff(tb1,tb2):
    """
    This function will tell you the difference in columns between two tables
    """
    schema1 = {f.name: str(f.dataType) for f in silver_tb_1.schema.fields}
    schema2 = {f.name: str(f.dataType) for f in silver_tb_2.schema.fields}
    all_column=set(schema1.keys()).union(schema2.keys())
    result=[]
    for col in all_column:
        data_type1=schema1.get(col,None)
        data_type2=schema2.get(col,None)
        in_tb1=col in schema1
        in_tb2=col in schema2

        if in_tb1 and in_tb2:
            if data_type1==data_type2:
                state='Match'
            else:
                state='No Match'
        else:
            state='Column Missing'
        result.append((col,data_type1,data_type2,in_tb1,in_tb2,state))
    col_diff_df=spark.createDataFrame(result,['column','data_type_tb1','data_type_tb2','in_tb1','in_tb2','state'])
    return col_diff_df

col_diff=column_diff(silver_tb_1,silver_tb_2)  
col_diff.display()    

column,data_type_tb1,data_type_tb2,in_tb1,in_tb2,state
y,StringType(),null,true,false,Column Missing
loan,StringType(),null,true,false,Column Missing
balance,IntegerType(),null,true,false,Column Missing
Item_Weight,null,DoubleType(),false,true,Column Missing
Item_MRP,null,DoubleType(),false,true,Column Missing
Outlet_Identifier,null,StringType(),false,true,Column Missing
Item_Type,null,StringType(),false,true,Column Missing
job,StringType(),null,true,false,Column Missing
contact,StringType(),null,true,false,Column Missing
Item_Identifier,null,StringType(),false,true,Column Missing


In [0]:
def null_counts(tb,label:str):
    """
    This function will tell you the null counts in each column of the table
    """
    print(f'{label} findings are:..')
    for i in tb.columns:
        null_count=tb.filter(col(i).isNull()).count()
        print(f'Null count in {i} is {null_count}')
null_counts(silver_tb_1,'bank')
null_counts(silver_tb_2,'BigMartSales')

bank findings are:..
Null count in age is 0
Null count in job is 0
Null count in marital is 0
Null count in education is 0
Null count in default is 0
Null count in balance is 0
Null count in housing is 0
Null count in loan is 0
Null count in contact is 0
Null count in day is 0
Null count in month is 0
Null count in duration is 0
Null count in campaign is 0
Null count in pdays is 0
Null count in previous is 0
Null count in poutcome is 0
Null count in y is 0
BigMartSales findings are:..
Null count in Item_Identifier is 0
Null count in Item_Weight is 1463
Null count in Item_Fat_Content is 0
Null count in Item_Visibility is 0
Null count in Item_Type is 0
Null count in Item_MRP is 0
Null count in Outlet_Identifier is 0
Null count in Outlet_Establishment_Year is 0
Null count in Outlet_Size is 2410
Null count in Outlet_Location_Type is 0
Null count in Outlet_Type is 0
Null count in Item_Outlet_Sales is 0


As we can see table 2 has 2 columns which has null values[Item_weight, Outlet_Size]

In [0]:
median_weight = silver_tb_2.approxQuantile("Item_Weight", [0.5], 0)[0]
silver_tb_2=silver_tb_2.fillna(median_weight, subset=["Item_Weight"])

In [0]:
mode_map = (
    silver_tb_2.filter("Outlet_Size IS NOT NULL")
      .groupBy("Outlet_Type", "Outlet_Size")
      .count()
      .withColumn("rn", F.row_number().over(
          Window.partitionBy("Outlet_Type").orderBy(F.desc("count"))
      ))

      .filter("rn = 1")
      .select("Outlet_Type", F.col("Outlet_Size").alias("mode"))
)

silver_tb_2=silver_tb_2.join(mode_map, "Outlet_Type", "left") \
       .withColumn("Outlet_Size", F.coalesce("Outlet_Size", "mode")) \
       .drop("mode")

In [0]:
null_counts(silver_tb_2,'BigMartSales')

BigMartSales findings are:..
Null count in Outlet_Type is 0
Null count in Item_Identifier is 0
Null count in Item_Weight is 0
Null count in Item_Fat_Content is 0
Null count in Item_Visibility is 0
Null count in Item_Type is 0
Null count in Item_MRP is 0
Null count in Outlet_Identifier is 0
Null count in Outlet_Establishment_Year is 0
Null count in Outlet_Size is 0
Null count in Outlet_Location_Type is 0
Null count in Item_Outlet_Sales is 0


In [0]:
def to_silver_schema(tb,label:str):
    """
    This function will store cleaned tables to silver schema
    """
    print(f'{label} is being stored...')
    (tb.write.mode('overwrite')
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(f'dev_catalog.silver_schema.{label}_silver_table'))
to_silver_schema(silver_tb_1,'bank')
to_silver_schema(silver_tb_2,'bigmartsales')

bank is being stored...
bigmartsales is being stored...


In [0]:
output_path='abfss://project-cont@1destorageacc.dfs.core.windows.net/silver'

In [0]:
silver_tb_1.write.mode('overwrite').option("header", "true").csv(output_path)

In [0]:
silver_tb_1.coalesce(1).write.format("csv") \
.option("header", True) \
.mode("overwrite") \
.save("abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Table1")

In [0]:
silver_tb_2.coalesce(1).write.format("csv") \
.option("header", True) \
.mode("overwrite") \
.save("abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Table2")

In [0]:
def Silver_To_adls(tb,path,tb_name:str):
    tb.coalesce(1).write.format("csv") \
    .option("header", True) \
    .mode("overwrite") \
    .save(path)
    print(f'{tb_name} is stored successfully at {path}')

In [0]:
Silver_To_adls(silver_tb_1,'abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Bank_silver','Table1_Bank')
Silver_To_adls(silver_tb_2,'abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Bigmart_silver','Table2_Bigmartsales')

Table1_Bank is stored successfully at abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Bank_silver
Table2_Bigmartsales is stored successfully at abfss://project-cont@1destorageacc.dfs.core.windows.net/silver/Bigmart_silver
